# 04. ESMFold로 접고 pLDDT 해석하기 🟡 (Colab GPU 권장)

**등급**: 🟡 Colab(T4 이상 GPU) · **다운로드**: `facebook/esmfold_v1` 가중치 약 8.4 GB (Colab 런타임에 저장됨) · **예상 시간(추정, 미검증)**: 모델 로드 5~10분 + 서열당 수 초

> ⚠️ **검증 범위**: 이 저장소의 Mac 환경에서는 8.4 GB 가중치를 받지 않았다. 대신 **무작위 초기화한 아주 작은 ESMFold**로
> `infer` → `output_to_pdb` → pLDDT 추출 → PDB 파싱의 **코드 경로만** 실행해 확인했다(출력 키, pLDDT shape `(1, L, 37)`, CA 인덱스 1, b-factor 일치).
> **실제 가중치로 접은 결과는 이 저장소에서 확인되지 않았으므로** 아래의 "예상"은 가설이며 직접 실행해 확인해야 한다.

**목표**: (1) 서열에서 구조를 예측하고 (2) 잔기별 **pLDDT**를 읽고 (3) 실험 구조와 비교(RMSD)하고 (4) 대조 서열(섞은 서열, 저복잡도 서열)에서 신뢰도가 어떻게 달라지는지 본다.

참고: 2026년에 ESM 팀(Biohub)이 ESMFold2를 공개했다 (모델 카드·프리프린트는 모듈 README 참고). 가중치가 훨씬 커서 이 노트북은 ESMFold v1을 쓴다.

In [ ]:
# Colab에서만 필요 (로컬에서는 SETUP.md의 환경을 사용):
# !pip -q install transformers accelerate biopython

import io
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from Bio.PDB import PDBParser, Superimposer
from transformers import EsmForProteinFolding

warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "GPU 런타임이 필요합니다 (Colab: 런타임 > 런타임 유형 변경 > T4 GPU)"
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

## 1. 모델 로드
- ESM-2 backbone을 fp16으로 두어 메모리를 줄인다 (`model.esm = model.esm.half()`).
- 긴 서열의 메모리를 줄이는 chunk 크기 설정 `model.trunk.set_chunk_size(64)`.

In [ ]:
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True).eval().to(DEVICE)
model.esm = model.esm.half()
model.trunk.set_chunk_size(64)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6))

## 2. 접기 함수와 pLDDT 추출
`model.infer(seq)`가 원시 출력 dict를 반환한다. `plddt`의 shape은 `(1, L, 37)`(atom37별)이고, 잔기 값은 **CA 원자(인덱스 1)**를 쓴다.
HF 구현의 pLDDT는 **0~1 범위**일 수 있으므로(범주형 혼합의 평균) 0~100 척도로 맞춰 표시한다.

In [ ]:
@torch.no_grad()
def fold(seq):
    out = model.infer(seq)
    plddt = out["plddt"][0, :, 1].float().cpu().numpy()          # CA 원자의 pLDDT
    assert plddt.max() <= 1.0 + 1e-3, "pLDDT 척도가 0~1이 아닙니다 — 라이브러리 버전이 바뀌었는지 확인하세요"
    plddt = plddt * 100                                           # 0~100 척도로 (HF 구현은 0~1을 반환함: 소스 확인)
    pae = out["predicted_aligned_error"][0].float().cpu().numpy() # (L, L) 잔기쌍 정렬 오차 예측 (단위·척도는 모델 문서 확인†)
    pdb = model.output_to_pdb(out)[0]
    ptm = float(out["ptm"])
    torch.cuda.empty_cache()
    return pdb, plddt, ptm, pae


UBQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"      # PDB 1UBQ, 76 aa
GB1 = "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"                            # PDB 1PGA, 56 aa
rng = np.random.RandomState(0)
SEQS = {
    "ubiquitin (1UBQ)": UBQ,
    "protein G B1 (1PGA)": GB1,
    "ubiquitin shuffled": "".join(rng.permutation(list(UBQ))),                               # 조성은 같고 순서만 섞음
    "low complexity (GS)n": "GS" * 38,
}
folded = {}
for name, s in SEQS.items():
    folded[name] = fold(s)
    pdb, pl, ptm, pae = folded[name]
    print(f"{name:24s} L={len(s):3d}  mean pLDDT={pl.mean():5.1f}  min={pl.min():5.1f}  pTM={ptm:.2f}  mean PAE={pae.mean():.2f}")

## 3. 잔기별 pLDDT

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for name, (pdb, pl, ptm, pae) in folded.items():
    ax.plot(pl, label=name, lw=1.5)
ax.axhline(70, color="gray", ls="--", lw=0.8)
ax.axhline(50, color="gray", ls=":", lw=0.8)
ax.set_xlabel("residue")
ax.set_ylabel("pLDDT (CA)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### PAE (잔기쌍 정렬 오차 예측)
pLDDT는 잔기별 국소 신뢰도이고, PAE는 **두 잔기의 상대 배치**에 대한 신뢰도다. 도메인 사이 배치는 pLDDT만으로 알 수 없고 PAE로 본다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, name in zip(axes, ["ubiquitin (1UBQ)", "protein G B1 (1PGA)", "ubiquitin shuffled"]):
    im = ax.imshow(folded[name][3], cmap="viridis_r")
    ax.set_title(name, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 4. 실험 구조와 비교 (CA RMSD)
RCSB에서 1UBQ, 1PGA 구조를 내려받아 예측 구조와 CA 원자로 겹쳐 RMSD를 잰다. 서열 길이가 같고 잔기 순서가 같다는 가정(두 구조 모두 해당)이다.

In [ ]:
def ca_atoms(structure):
    return [r["CA"] for r in structure[0]["A"] if r.id[0] == " " and "CA" in r]


def rmsd_to_experiment(pid, pred_pdb):
    path = f"{pid}.pdb"
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.pdb", path)
    exp = PDBParser(QUIET=True).get_structure("exp", path)
    pred = PDBParser(QUIET=True).get_structure("pred", io.StringIO(pred_pdb))
    pred_ca = [r["CA"] for r in pred[0].get_residues() if "CA" in r]
    exp_ca = ca_atoms(exp)
    assert len(exp_ca) == len(pred_ca), (len(exp_ca), len(pred_ca))
    sup = Superimposer()
    sup.set_atoms(exp_ca, pred_ca)
    return sup.rms


for name, pid in [("ubiquitin (1UBQ)", "1UBQ"), ("protein G B1 (1PGA)", "1PGA")]:
    print(f"{name:22s} CA RMSD vs experiment: {rmsd_to_experiment(pid, folded[name][0]):.2f} Å")

## 5. 점 돌연변이의 영향 — 한 서열의 변이 몇 개
유비퀴틴의 무작위 점돌연변이 8개를 접고 평균 pLDDT 변화를 본다.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
wt_pl = folded["ubiquitin (1UBQ)"][1]
W = 5                                                                 # 변이 위치 ±5 잔기 창
rows = []
for _ in range(8):
    i = int(rng.randint(len(UBQ)))
    new = rng.choice([a for a in AA if a != UBQ[i]])
    mut = UBQ[:i] + new + UBQ[i + 1:]
    _, pl, _, _ = fold(mut)
    lo, hi = max(0, i - W), min(len(UBQ), i + W + 1)
    rows.append((f"{UBQ[i]}{i + 1}{new}", pl.mean() - wt_pl.mean(), pl[lo:hi].mean() - wt_pl[lo:hi].mean()))
for m, d_all, d_loc in rows:
    print(f"{m:6s} Δ 평균 pLDDT(전체) = {d_all:+.2f} | Δ 평균 pLDDT(변이 ±{W}) = {d_loc:+.2f}")

## 6. 해석 가이드 (직접 실행한 결과로 확인할 것)
**예상(가설)** — 이 저장소에서 실제 가중치로 확인되지 않았다:
- 자연 단백질(유비퀴틴, GB1)은 높은 pLDDT를 보이고 실험 구조와 RMSD가 작을 것이다.
- 서열을 섞은 대조군과 `(GS)n` 저복잡도 서열은 낮은 pLDDT가 나올 가능성이 크다. 단, **낮은 pLDDT = 무질서의 증명이 아니라 모델이 확신하지 못한다는 뜻**이다.
- 점돌연변이는 서열 전체 평균 pLDDT를 크게 바꾸지 못할 것이다. 다만 변이 효과는 **국소적**일 수 있어 전체 평균 대신 변이 주변 창(±5)의 pLDDT와 PAE도 함께 본다. 구조 예측기의 신뢰도는 변이의 기능적 영향을 직접 측정하지 않는다.

**읽는 법**
- pLDDT 구간별 의미는 모델 문서의 규약을 확인하라. 흔히 70 이상을 비교적 신뢰, 50 미만을 낮은 신뢰로 보지만 **척도(0~1 vs 0~100)를 먼저 확인**한다.
- pTM·PAE는 도메인 배치와 전체 폴드의 신뢰도를 다룬다(이 노트북은 pTM, 평균 PAE, PAE 행렬을 출력한다). pLDDT만으로 도메인 상대 배치는 알 수 없다.
- 단일 서열 모델(ESMFold)은 MSA 기반 모델과 다른 오류 패턴을 보인다. 사전학습 데이터에 포함된 단백질에서는 특히 잘 맞을 수 있다 (평가–사전학습 겹침, 모듈 01).

## 7. 한계와 과제
- 구조 하나를 정확히 예측한다는 것이 변이 효과, 동역학, 상호작용을 이해한다는 뜻은 아니다.
- 과제: [../exercises.md](../exercises.md) 과제 5 (ProteinGym 소형 assay에서 pLDDT 변화와 DMS 점수의 상관).